# 02 Shape Target Diagnostics

This notebook turns the refreshed observed Dutch quarter-hour DAM data into the canonical within-hour shape target used by later modelling stages.

Current scope:

- confirm that the shared cleaned quarterly source and the dedicated continuation source remain aligned;
- identify complete four-quarter hour groups after cleaning;
- construct the hourly mean and zero-mean within-hour delta target;
- define the empirical train/validation/test split from the actually available observed range.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path
    for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
    if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)

PACKAGE_ROOT = REPO_ROOT / "scripts" / "Data" / "02_Forecasting" / "01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from quarterhour_da import (
    QuarterHourDAExtensionConfig,
    assert_thesis_grade_actual_source_authorized,
    build_thesis_grade_frozen_actual_metadata,
    find_frozen_actual_version,
    find_latest_canonical_actual_run,
    find_latest_observed_deterministic_run,
    find_latest_phase01_run,
    find_latest_phase02_run,
    find_latest_phase03_run,
    find_latest_phase04_run,
    find_latest_phase07_run,
    find_latest_phase07_upstream_refresh_run,
    load_frozen_actual_diagnostics,
    load_frozen_actual_manifest,
    load_frozen_actual_path,
    resolve_frozen_actual_registry_entry,
    run_observed_market_deterministic_forecast,
)

config = QuarterHourDAExtensionConfig()
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
plt.style.use("seaborn-v0_8-whitegrid")

## Optional Phase 2 Runner

The notebook loads the latest saved Phase 2 artifact by default. Set `RUN_PHASE02 = True` only when you want to rebuild the shape-target outputs from inside the notebook.

In [ ]:
RUN_PHASE02 = False

if RUN_PHASE02:
    command = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "Data" / "02_Forecasting" / "01_DA_prices" / "run_15min_phase02_shape_targets.py"),
    ]
    completed = subprocess.run(command, cwd=REPO_ROOT, capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"Phase 2 shape-target build failed with exit code {completed.returncode}.")

In [ ]:
latest_run = find_latest_phase02_run(config)
if latest_run is None:
    raise FileNotFoundError("No saved Phase 2 artifact exists yet. Run the phase 2 script first.")

source_availability = pd.read_csv(latest_run / "source_availability_summary.csv")
source_consistency = pd.read_csv(latest_run / "source_consistency_summary.csv")
hour_group_summary = pd.read_csv(latest_run / "hour_group_summary.csv")
split_summary = pd.read_csv(latest_run / "split_summary.csv")
quarter_diag = pd.read_csv(latest_run / "quarter_diagnostics.csv")
hour_quarter_diag = pd.read_csv(latest_run / "hour_quarter_diagnostics.csv")
shape_target = pd.read_csv(latest_run / "shape_target_long.csv")
run_summary = json.loads((latest_run / "run_summary.json").read_text(encoding="utf-8"))

display(pd.DataFrame([{"latest_phase02_run": str(latest_run)}]))

## Source Availability

In [ ]:
display(source_availability)

## Shared vs Companion Source Consistency

In [ ]:
display(source_consistency)

## Complete Four-Quarter Hour Groups

In [ ]:
complete_summary = (
    hour_group_summary.groupby("hour_group_quality", dropna=False)
    .agg(hour_groups=("hour_start_utc", "size"))
    .reset_index()
    .sort_values("hour_group_quality")
)
display(complete_summary)

## Empirical Split Summary

In [ ]:
display(split_summary)

## Quarter-Level Delta Diagnostics

In [ ]:
display(quarter_diag)

## Delta Distribution By Quarter

The boxplot shows whether some quarter positions tend to sit above or below the within-hour mean.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
shape_target.boxplot(column="delta_eur_per_mwh", by="quarter_index", ax=ax, grid=False)
ax.set_title("Delta Distribution By Quarter")
ax.set_xlabel("Quarter index")
ax.set_ylabel("Delta (EUR/MWh)")
fig.suptitle("")
plt.show()

## Hour-Quarter Heatmap

The heatmap reports the average within-hour deviation by local hour of day and quarter index.

In [ ]:
heatmap = hour_quarter_diag.pivot(index="local_hour_of_day", columns="quarter_index", values="mean_delta")
fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(heatmap.values, aspect="auto", cmap="coolwarm")
ax.set_xticks(range(len(heatmap.columns)))
ax.set_xticklabels([str(value) for value in heatmap.columns])
ax.set_yticks(range(len(heatmap.index)))
ax.set_yticklabels([str(value) for value in heatmap.index])
ax.set_xlabel("Quarter index")
ax.set_ylabel("Local hour of day")
ax.set_title("Mean Delta By Hour And Quarter")
fig.colorbar(im, ax=ax, label="Mean delta (EUR/MWh)")
plt.show()

## Zero-Mean Check

In [ ]:
zero_mean_check = (
    shape_target.groupby("hour_start_utc")["delta_eur_per_mwh"]
    .mean()
    .abs()
    .agg(["max", "mean"])
    .rename({"max": "max_abs_hourly_delta_mean", "mean": "mean_abs_hourly_delta_mean"})
    .to_frame()
    .T
)
display(zero_mean_check)